# 淘宝用户行为分析
本 Notebook 使用 CSV 或者 MySQL 数据，完成：流量与转化、漏斗、留存与复购、RFM 分层和商品分析。每个分析单元包含结论与业务建议。

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import os

plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False
sns.set_theme(style='whitegrid')

In [ ]:
# 修改为你的 CSV 路径或数据库连接字符串
csv_path = os.path.abspath(os.path.join('..','data','UserBehavior.csv'))
print('CSV path:', csv_path)
df = pd.read_csv(csv_path, names=['user_id','item_id','category_id','behavior_type','timestamp'], nrows=1000000)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
# 规范化行为类型（支持 pv/fav/cart/buy 与 1/2/3/4）
df['behavior_type'] = df['behavior_type'].astype(str).str.strip().str.lower()
behavior_map = {'pv':'浏览','fav':'收藏','cart':'加购','buy':'购买','1':'浏览','2':'收藏','3':'加购','4':'购买'}
df['behavior_name'] = df['behavior_type'].map(behavior_map)
df.head()

## 流量与活跃（PV/UV、日活、小时活跃）
计算 PV/UV，并绘制按日期与小时的活跃趋势。结论后给出运营建议。

In [ ]:
# PV / UV / 日活（按 date）
pv = len(df[df['behavior_name']=='浏览'])
uv = df['user_id'].nunique()
daily = df.groupby('date').agg(pv_total=('behavior_name', lambda x: (x=='浏览').sum()),
                                         uv_daily=('user_id', 'nunique'))
daily = daily.reset_index()
print('PV:', pv, 'UV:', uv)
# 绘制日活趋势
os.makedirs(os.path.join('..','charts'), exist_ok=True)
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(pd.to_datetime(daily['date']), daily['pv_total'], marker='o')
ax.set_title('日 PV 趋势')
ax.set_ylabel('PV')
fig.savefig(os.path.abspath(os.path.join('..','charts','daily_pv.png')), dpi=150, bbox_inches='tight')
plt.show()
# 小时级活跃
hourly = df.groupby('hour').agg(pv=('behavior_name', lambda x: (x=='浏览').sum()))
hourly = hourly.reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(10,3))
sns.barplot(x=hourly.index, y=hourly['pv'], ax=ax, color='skyblue')
ax.set_title('小时 PV 分布')
fig.savefig(os.path.abspath(os.path.join('..','charts','hourly_pv.png')), dpi=150, bbox_inches='tight')
plt.show()

## 浏览→加购→收藏→购买 漏斗及跳失率
使用行为序列近似计算各环节人数与转化率。跳失率这里用当天仅有一次行为的用户占比作为近似。

In [ ]:
# 漏斗统计（按行为名称）
pv = (df['behavior_name']=='浏览').sum()
cart = (df['behavior_name']=='加购').sum()
fav = (df['behavior_name']=='收藏').sum()
buy = (df['behavior_name']=='购买').sum()
print('pv', pv, 'cart', cart, 'fav', fav, 'buy', buy)
# 跳失率（当天只有一次行为的用户占比）
actions_per_user_day = df.groupby(['user_id','date']).size().reset_index(name='actions')
bounce_rate = (actions_per_user_day['actions']==1).sum() / len(actions_per_user_day)
print(f'跳失率（近似）: {bounce_rate:.2%}')
# 绘制简单漏斗
funnel = pd.DataFrame({'stage':['浏览','收藏/加购','购买'], 'count':[pv, fav+cart, buy]})
fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(data=funnel, x='stage', y='count', palette='Blues_d', ax=ax)
ax.set_title('用户行为转化漏斗')
fig.savefig(os.path.abspath(os.path.join('..','charts','funnel.png')), dpi=150, bbox_inches='tight')
plt.show()

## 留存分析（Cohort）
构建用户的首日，并计算后续 0-30 天的留存率曲线。Notebook 中用 pandas 演示；完整 SQL 脚本见 `sql/02_retention.sql`。

In [ ]:
# Cohort 留存（pandas 实现，示例）
first = df.groupby('user_id')['date'].min().reset_index().rename(columns={'date':'first_date'})
events = df[['user_id','date']].drop_duplicates()
cohort = events.merge(first, on='user_id')
cohort['days_from_first'] = (pd.to_datetime(cohort['date']) - pd.to_datetime(cohort['first_date'])).dt.days
cohort_counts = cohort.groupby(['first_date','days_from_first'])['user_id'].nunique().reset_index()
# 构建留存表（示例显示次日/7日/30日留存）
pivot = cohort_counts.pivot(index='first_date', columns='days_from_first', values='user_id').fillna(0)
pivot['cohort_size'] = pivot[0]
for day in [1,7,30]:
    if day in pivot.columns:
        pivot[f'retention_{day}'] = pivot[day] / pivot['cohort_size']
    else:
        pivot[f'retention_{day}'] = np.nan
pivot_retention = pivot[[c for c in pivot.columns if str(c).startswith('retention')]]
pivot_retention.head()

## 复购分析
计算复购率与复购周期分布。复购定义：同一用户有 2 次及以上购买行为。

In [ ]:
# 复购率与周期
orders = df[df['behavior_name']=='购买'].copy()
orders['order_date'] = pd.to_datetime(orders['date'])
order_counts = orders.groupby('user_id').agg(order_times=('order_date','nunique'), first_order=('order_date','min'), last_order=('order_date','max')).reset_index()
repurchase_rate = (order_counts['order_times']>=2).mean()
print(f'复购率: {repurchase_rate:.2%}')
# 复购周期分布（针对有>=2单的用户）
multi = orders.groupby('user_id').agg(dates=('order_date', lambda x: sorted(x.unique()))).reset_index()
def first_interval(dates):
    if len(dates)<2: return np.nan
    return (dates[1]-dates[0]).days
multi['first_interval'] = multi['dates'].apply(first_interval)
multi['first_interval'].dropna().describe()

## RFM 分层与 KMeans 验证
使用购买数据计算 R（Recency）、F（Frequency）、M（Monetary，若无金额可用次数/订单替代），并使用 KMeans 聚类验证分层。

In [ ]:
# 构建 RFM（此数据集无金额字段，使用购买次数替代 M）
today = pd.to_datetime(df['datetime'].max()).normalize() + pd.Timedelta(days=1)
orders = df[df['behavior_name']=='购买'].copy()
rfm = orders.groupby('user_id').agg(recency_days=('datetime', lambda x: (today - x.max()).days), frequency=('item_id','count')).reset_index()
# 对数/归一化后 KMeans
rfm['recency_days'] = rfm['recency_days'].astype(float)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(rfm[['recency_days','frequency']].fillna(0))
kmeans = KMeans(n_clusters=4, random_state=42).fit(X)
rfm['cluster'] = kmeans.labels_
rfm.groupby('cluster').agg({'recency_days':'median','frequency':'median','user_id':'count'})

## 商品与品类分析
Top 商品 / 品类排行，以及简单的关联（频繁一起购买的品类组合）。

In [ ]:
top_items = df[df['behavior_name']=='购买']['item_id'].value_counts().head(20)
top_items.plot(kind='bar', figsize=(10,4))
plt.title('Top 20 购买商品')
plt.tight_layout()
plt.savefig(os.path.abspath(os.path.join('..','charts','top_items.png')), dpi=150, bbox_inches='tight')
plt.show()

---
## 下一步建议
- 将 CSV 导入 MySQL，用 SQL 脚本完成更大规模计算（见 `sql/`）。
- 基于 RFM 与 KMeans 的分群，写出每组的运营策略并在 README 中呈现。
- 使用 Power BI 连接 `charts/` 输出和 SQL 查询，制作交互式看板。